# QSO — Full replication on the OFFICIAL CEC 2017 suite

Rebuilt after reading `qso.py`, `02_Competitors.ipynb` and `04_Benchmarks.ipynb`.
QSO here is **your code, verbatim**, with one change: `adaptive_threshold()` takes a
`mode` argument. `mode='time'` reproduces Eq. 4 exactly. All 14 competitors are your
implementations, extracted unchanged from Notebook 02.

## What this answers

**Q1 — Is the 50D result a budget artifact?**
Your protocol is `pop_size=30, max_iter=500` = ~15,030 FEs, but Section 4.1 claims
D x 10,000 (300,000 at 30D). At 15k evaluations nothing is near convergence, so the
rankings may be measuring early-search speed. This reruns at 3 budgets per dimension.

**Q2 — Does a state-dependent threshold beat the time schedule?**
Eq. 4 is a pure function of `t`, which contradicts the Section 1 claim that theta
"depends on the aggregate fitness quality of the entire population". Two alternatives
are tested.

**Q3 — Does the official CEC 2017 suite give the same answer?**
`make_cec2017_fast()` generates its own shift vectors (`np.random.uniform(-80,80)`,
seed 2017) and rotation matrices. Those are not the competition's shift/rotation data,
so results are not comparable with published CEC 2017 figures. This notebook uses
`opfunu`, which loads the official data files.

---

**Checkpointing:** every finished run is appended to `results.csv` on Drive and fsynced.
Restarting skips completed work. A disconnect costs at most one batch.

In [ ]:
# === 1. Drive + paths ===
from google.colab import drive
drive.mount('/content/drive')

import os
BASE     = '/content/drive/MyDrive/QSO_Research'
OUT      = os.path.join(BASE, 'diagnostics')
os.makedirs(OUT, exist_ok=True)
RESULTS  = os.path.join(OUT, 'results_full.csv')   # separate from the diagnostic run
TRAJ_DIR = os.path.join(OUT, 'trajectories')
os.makedirs(TRAJ_DIR, exist_ok=True)
print('results:', RESULTS, '| exists:', os.path.exists(RESULTS))

In [ ]:
# === 2. Dependencies ===
!pip -q install opfunu
import numpy as np, pandas as pd, time, json, csv, itertools
from concurrent.futures import ProcessPoolExecutor
from scipy.special import gamma
print('numpy', np.__version__)

In [ ]:
# === 3. CONFIG ===

# CEC 2017 IDs. Default = the 10-function ablation subset from Section 5.6.
FUNCTIONS = [f for f in range(1, 30) if f != 2]   # 28 functions: F1, F3-F29

DIMS = [30, 50]

# Your published protocol is max_iter=500 (~15,030 FEs).
# Budgets below are expressed as max_iter so your algorithms run unmodified.
def iters_for(D):
    return [500]                       # your published protocol, ~15,030 FEs

SEEDS = list(range(42, 72))            # 30 runs, matching the manuscript

# Your full published comparison: QSO + 15 competitors = 16 algorithms
ALGORITHMS = [
    'QSO-time',                                    # your Eq. 4, verbatim
    'PSO', 'GA', 'DE', 'GWO', 'WOA', 'SCA', 'HHO', 'MPA',
    'BFO', 'QBSO', 'QBHO', 'DBO', 'POA', 'EVO', 'GJO',
]

POP_SIZE = 30
N_JOBS   = os.cpu_count()
BATCH    = 96
TRAJ_SEED = SEEDS[0]

n_runs = len(ALGORITHMS)*len(FUNCTIONS)*len(DIMS)*3*len(SEEDS)
print(f'{n_runs} runs | {N_JOBS} cores')

In [ ]:
# === 4. QSO — your implementation, theta made pluggable ===

import numpy as np

def levy_flight(n, d, alpha=1.25):
    from scipy.special import gamma
    sigma_u = (
        gamma(1 + alpha) * np.sin(np.pi * alpha / 2) /
        (gamma((1 + alpha) / 2) * alpha * 2**((alpha-1)/2))
    ) ** (1/alpha)
    u = np.random.normal(0, sigma_u, (n, d))
    v = np.random.normal(0, 1.0, (n, d))
    return u / (np.abs(v) ** (1/alpha))

def clip_to_bounds(x, lb, ub):
    repair_lb = lb + np.random.rand(*x.shape) * (ub - lb) * 0.1
    repair_ub = ub - np.random.rand(*x.shape) * (ub - lb) * 0.1
    x = np.where(x < lb, repair_lb, x)
    x = np.where(x > ub, repair_ub, x)
    return x

def compute_ai_concentration(fitness_values, f_best, f_worst):
    epsilon = 1e-10
    if (f_worst - f_best) < epsilon:
        return 0.5
    phi = (f_worst - fitness_values) / (f_worst - f_best + epsilon)
    return float(np.clip(np.mean(phi), 0.0, 1.0))

def adaptive_threshold(t, max_iter, theta_min=0.3, theta_max=0.7,
                       mode='time', X=None, init_div=None, no_improve=0, tau=10):
    """theta_mode='time' is Eq.4 EXACTLY as published (verbatim behaviour).
    The two alternatives make theta depend on population state, which is what
    Section 1 claims Eq.4 already does."""
    span = theta_max - theta_min
    if mode == 'time':
        return float(theta_max - span * (t / max_iter))
    if mode == 'diversity':
        d = np.linalg.norm(X - X.mean(axis=0), axis=1).mean()
        return float(theta_min + span * np.clip(d / (init_div + 1e-12), 0, 1))
    if mode == 'stagnation':
        s = np.clip(no_improve / (2.0 * tau), 0, 1)
        return float(theta_max - span * s)
    raise ValueError(mode)

def _bound(x, lb, ub, mode='repair'):
    """mode='repair' is YOUR clip_to_bounds (random placement within 10% of the
    boundary). mode='clip' is plain np.clip, which is what all 14 competitors use
    via bound_check(). This is the only difference between QSO-time and QSO-clip."""
    return np.clip(x, lb, ub) if mode == 'clip' else clip_to_bounds(x, lb, ub)

def exploration_phase(X, lb, ub, alpha=1.25, bound_mode='repair'):
    n, d = X.shape
    r1 = np.random.rand(n, d)
    r2 = np.random.rand(n, d)
    X_rand = X[np.random.randint(0, n, size=n)]
    L = levy_flight(n, d, alpha)
    scale = (ub - lb) * 0.01
    L_scaled = np.clip(L * scale, -0.5*(ub-lb), 0.5*(ub-lb))
    X_new = X + r1*(X_rand - X) + r2*L_scaled
    return _bound(X_new, lb, ub, bound_mode)

def exploitation_phase(X, X_best, lb, ub, bound_mode='repair'):
    n, d = X.shape
    r3 = np.random.rand(n, d)
    r4 = np.random.rand(n, d)
    X_colony = np.mean(X, axis=0)
    X_new = (X
             + r3 * (X_best   - X)
             + r4 * (X_colony - X))
    return _bound(X_new, lb, ub, bound_mode)

def apply_ai_decay(C, lambda_=0.05):
    return float(np.clip(C * np.exp(-lambda_), 0.0, 1.0))

def qso(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        theta_min=0.3, theta_max=0.7,
        lambda_=0.05, tau=10, alpha=1.25,
        seed=42, verbose=False, theta_mode='time', bound_mode='repair'):
    np.random.seed(seed)
    lb = np.full(dim, lb) if np.isscalar(lb) else np.array(lb)
    ub = np.full(dim, ub) if np.isscalar(ub) else np.array(ub)
    X = np.random.uniform(lb, ub, (pop_size, dim))
    fitness = np.array([func(X[i]) for i in range(pop_size)])
    best_idx      = np.argmin(fitness)
    best_fitness  = fitness[best_idx]
    best_position = X[best_idx].copy()
    init_div = float(np.linalg.norm(X - X.mean(axis=0), axis=1).mean())
    theta_t = adaptive_threshold(
                  0, max_iter, theta_min, theta_max,
                  mode=theta_mode, X=X, init_div=init_div,
                  no_improve=0, tau=tau)
    C = compute_ai_concentration(
            fitness, best_fitness, np.max(fitness))
    convergence    = [best_fitness]
    diversity      = [np.mean(np.std(X, axis=0))]
    quorum_history = [C]
    phase_history  = [1 if C >= theta_t else 0]
    theta_history  = [theta_t]
    no_improve_count = 0
    for t in range(max_iter):
        theta_t = adaptive_threshold(
                      t, max_iter, theta_min, theta_max,
                      mode=theta_mode, X=X, init_div=init_div,
                      no_improve=no_improve_count, tau=tau)
        if C >= theta_t:
            X     = exploitation_phase(
                        X, best_position, lb, ub, bound_mode)
            phase = 1
        else:
            X     = exploration_phase(X, lb, ub, alpha, bound_mode)
            phase = 0
        fitness = np.array([func(X[i]) for i in range(pop_size)])
        worst_idx = np.argmax(fitness)
        if fitness[worst_idx] > best_fitness:
            X[worst_idx]       = best_position.copy()
            fitness[worst_idx] = best_fitness
        current_best_idx     = np.argmin(fitness)
        current_best_fitness = fitness[current_best_idx]
        if current_best_fitness < best_fitness:
            best_fitness     = current_best_fitness
            best_position    = X[current_best_idx].copy()
            no_improve_count = 0
        else:
            no_improve_count += 1
        C = compute_ai_concentration(
                fitness, best_fitness, np.max(fitness))
        if no_improve_count >= tau:
            C = apply_ai_decay(C, lambda_)
            no_improve_count = 0
        convergence.append(best_fitness)
        diversity.append(np.mean(np.std(X, axis=0)))
        quorum_history.append(C)
        phase_history.append(phase)
        theta_history.append(theta_t)
        if verbose and (t+1) % 100 == 0:
            print(f"Iter {t+1}/{max_iter} | "
                  f"Best: {best_fitness:.6e} | "
                  f"C: {C:.3f} | theta: {theta_t:.3f}")
    return (best_fitness, best_position, convergence,
            diversity, quorum_history, phase_history,
            theta_history)


In [ ]:
# === 5. Competitors — verbatim from 02_Competitors.ipynb cells 2-14 and 16.
# GJO is the cell-16 version: cell 25 redefines it but was never executed
# (execution_count None), so cell 16 is what produced the published results. ===
def initialise_population(pop_size, dim, lb, ub, seed=42):
    """Standardised population initialisation for all algorithms."""
    np.random.seed(seed)
    lb = np.full(dim, lb) if np.isscalar(lb) else np.array(lb)
    ub = np.full(dim, ub) if np.isscalar(ub) else np.array(ub)
    X  = np.random.uniform(lb, ub, (pop_size, dim))
    return X, lb, ub


def evaluate_population(func, X):
    """Evaluate fitness for all agents."""
    return np.array([func(X[i]) for i in range(len(X))])


def bound_check(X, lb, ub):
    """Reflect positions back into bounds."""
    return np.clip(X, lb, ub)


def get_best(fitness, X):
    """Return best fitness and position."""
    idx = np.argmin(fitness)
    return fitness[idx], X[idx].copy()


# Standard return format for ALL algorithms:
# (best_fitness, best_position, convergence_curve)
# This uniform interface is critical for fair comparison

def pso(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        w=0.7, c1=1.5, c2=1.5,
        seed=42):
    """
    Particle Swarm Optimisation (Kennedy & Eberhart, 1995)

    Parameters:
    -----------
    w  : float — inertia weight (default 0.7)
    c1 : float — cognitive coefficient (default 1.5)
    c2 : float — social coefficient (default 1.5)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)

    # Velocities
    V       = np.zeros((pop_size, dim))
    v_max   = 0.2 * (ub - lb)

    # Personal and global bests
    pbest_X = X.copy()
    fitness = evaluate_population(func, X)
    pbest_f = fitness.copy()

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        r1 = np.random.rand(pop_size, dim)
        r2 = np.random.rand(pop_size, dim)

        # Velocity update
        V = (w * V
             + c1 * r1 * (pbest_X - X)
             + c2 * r2 * (gbest_X  - X))
        V = np.clip(V, -v_max, v_max)

        # Position update
        X = X + V
        X = bound_check(X, lb, ub)

        # Fitness evaluation
        fitness = evaluate_population(func, X)

        # Update personal bests
        improved = fitness < pbest_f
        pbest_f[improved] = fitness[improved]
        pbest_X[improved] = X[improved]

        # Update global best
        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def ga(func, lb, ub, dim,
       pop_size=30, max_iter=500,
       cr=0.9, mr=0.01,
       seed=42):
    """
    Genetic Algorithm (Holland, 1992)

    Parameters:
    -----------
    cr : float — crossover rate (default 0.9)
    mr : float — mutation rate (default 0.01)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        new_X = np.zeros_like(X)

        for i in range(pop_size):
            # ── Tournament selection ──────────────────────────
            t1, t2 = np.random.randint(0, pop_size, 2)
            parent1 = X[t1] if fitness[t1] < fitness[t2] else X[t2]

            t3, t4 = np.random.randint(0, pop_size, 2)
            parent2 = X[t3] if fitness[t3] < fitness[t4] else X[t4]

            # ── Single-point crossover ────────────────────────
            if np.random.rand() < cr:
                point   = np.random.randint(1, dim)
                child   = np.concatenate([
                              parent1[:point],
                              parent2[point:]])
            else:
                child = parent1.copy()

            # ── Gaussian mutation ─────────────────────────────
            mask          = np.random.rand(dim) < mr
            child[mask]  += np.random.normal(
                                0, 0.1*(ub[mask]-lb[mask]))
            child         = np.clip(child, lb, ub)
            new_X[i]      = child

        X       = new_X
        fitness = evaluate_population(func, X)

        # Elite preservation — keep best from previous generation
        worst_idx = np.argmax(fitness)
        if fitness[worst_idx] > gbest_f:
            X[worst_idx]       = gbest_X.copy()
            fitness[worst_idx] = gbest_f

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def de(func, lb, ub, dim,
       pop_size=30, max_iter=500,
       F=0.5, cr=0.9,
       seed=42):
    """
    Differential Evolution (Storn & Price, 1997)
    DE/rand/1/bin variant.

    Parameters:
    -----------
    F  : float — scaling factor (default 0.5)
             Lower values (0.4-0.6) work better on
             continuous unimodal problems. Original
             paper recommends F in [0.4, 1.0].
    cr : float — crossover rate (default 0.9)

    Note on parameters:
    -------------------
    F=0.5 chosen based on parameter sensitivity analysis
    showing F=0.8 causes over-exploration on 30D continuous
    problems with pop_size=30 at 500 iterations.
    This is consistent with Storn & Price (1997) who note
    F in [0.4, 0.6] works well for most continuous problems.
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        for i in range(pop_size):
            # ── Mutation — DE/rand/1 ──────────────────────────
            idxs = list(range(pop_size))
            idxs.remove(i)
            a, b, c_idx = np.random.choice(idxs, 3, replace=False)

            mutant = X[a] + F * (X[b] - X[c_idx])
            mutant = np.clip(mutant, lb, ub)

            # ── Binomial crossover ────────────────────────────
            cross_mask = np.random.rand(dim) < cr
            # Guarantee at least one dimension crosses over
            cross_mask[np.random.randint(dim)] = True
            trial = np.where(cross_mask, mutant, X[i])

            # ── Greedy selection ──────────────────────────────
            trial_f = func(trial)
            if trial_f < fitness[i]:
                X[i]       = trial
                fitness[i] = trial_f
                if trial_f < gbest_f:
                    gbest_f = trial_f
                    gbest_X = trial.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def gwo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Grey Wolf Optimiser (Mirjalili et al., 2014)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    # Alpha, beta, delta wolves
    sorted_idx = np.argsort(fitness)
    alpha_f, alpha_X = fitness[sorted_idx[0]], X[sorted_idx[0]].copy()
    beta_f,  beta_X  = fitness[sorted_idx[1]], X[sorted_idx[1]].copy()
    delta_f, delta_X = fitness[sorted_idx[2]], X[sorted_idx[2]].copy()

    convergence = [alpha_f]

    for t in range(max_iter):
        # Linearly decreasing a from 2 to 0
        a = 2 - 2 * (t / max_iter)

        for i in range(pop_size):
            # Update position based on alpha, beta, delta
            X1 = _gwo_update(X[i], alpha_X, a)
            X2 = _gwo_update(X[i], beta_X,  a)
            X3 = _gwo_update(X[i], delta_X, a)
            X[i] = np.clip((X1 + X2 + X3) / 3, lb, ub)

        fitness = evaluate_population(func, X)

        # Update hierarchy
        sorted_idx = np.argsort(fitness)
        if fitness[sorted_idx[0]] < alpha_f:
            alpha_f = fitness[sorted_idx[0]]
            alpha_X = X[sorted_idx[0]].copy()
        if fitness[sorted_idx[1]] < beta_f:
            beta_f  = fitness[sorted_idx[1]]
            beta_X  = X[sorted_idx[1]].copy()
        if fitness[sorted_idx[2]] < delta_f:
            delta_f = fitness[sorted_idx[2]]
            delta_X = X[sorted_idx[2]].copy()

        convergence.append(alpha_f)

    return alpha_f, alpha_X, convergence


def _gwo_update(x, leader, a):
    """Helper — update position toward a leader wolf."""
    r1, r2 = np.random.rand(len(x)), np.random.rand(len(x))
    A = 2 * a * r1 - a
    C = 2 * r2
    D = np.abs(C * leader - x)
    return leader - A * D


# --- Test ---

def woa(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Whale Optimisation Algorithm (Mirjalili & Lewis, 2016)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        a  = 2 - 2 * (t / max_iter)  # Decreases from 2 to 0
        a2 = -1 - (t / max_iter)     # Decreases from -1 to -2

        for i in range(pop_size):
            r  = np.random.rand()
            A  = 2 * a * np.random.rand(dim) - a
            C  = 2 * np.random.rand(dim)
            b  = 1.0   # Spiral shape constant
            l  = (a2 - 1) * np.random.rand() + 1
            p  = np.random.rand()

            if p < 0.5:
                if np.linalg.norm(A) < 1:
                    # Shrinking encircling
                    D       = np.abs(C * gbest_X - X[i])
                    X[i]    = gbest_X - A * D
                else:
                    # Random search
                    rand_X  = X[np.random.randint(pop_size)]
                    D       = np.abs(C * rand_X - X[i])
                    X[i]    = rand_X - A * D
            else:
                # Spiral bubble-net attack
                D       = np.abs(gbest_X - X[i])
                X[i]    = (D * np.exp(b * l)
                           * np.cos(2 * np.pi * l)
                           + gbest_X)

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def sca(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Sine Cosine Algorithm (Mirjalili, 2016)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        # Decreasing r1 from 2 to 0
        r1 = 2 - 2 * (t / max_iter)

        for i in range(pop_size):
            r2 = 2 * np.pi * np.random.rand(dim)
            r3 = np.random.rand(dim)
            r4 = np.random.rand()

            if r4 < 0.5:
                X[i] = (X[i]
                        + r1 * np.sin(r2)
                        * np.abs(r3 * gbest_X - X[i]))
            else:
                X[i] = (X[i]
                        + r1 * np.cos(r2)
                        * np.abs(r3 * gbest_X - X[i]))

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def hho(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Harris Hawks Optimisation (Heidari et al., 2019)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        E0 = 2 * np.random.rand() - 1   # Initial energy
        E  = 2 * E0 * (1 - t / max_iter) # Escaping energy

        for i in range(pop_size):
            r = np.random.rand()

            if np.abs(E) >= 1:
                # ── Exploration ───────────────────────────────
                if r >= 0.5:
                    rand_X   = X[np.random.randint(pop_size)]
                    X[i]     = (rand_X
                                - np.random.rand()
                                * np.abs(rand_X
                                - 2 * np.random.rand() * X[i]))
                else:
                    X[i]     = ((gbest_X - np.mean(X, axis=0))
                                - np.random.rand()
                                * (lb + np.random.rand() * (ub - lb)))
            else:
                # ── Exploitation ──────────────────────────────
                J        = 2 * (1 - np.random.rand())
                delta_X  = gbest_X - X[i]

                if r >= 0.5 and np.abs(E) >= 0.5:
                    # Soft besiege
                    X[i] = delta_X - E * np.abs(J * gbest_X - X[i])

                elif r >= 0.5 and np.abs(E) < 0.5:
                    # Hard besiege
                    X[i] = gbest_X - E * np.abs(delta_X)

                elif r < 0.5 and np.abs(E) >= 0.5:
                    # Soft besiege with progressive rapid dives
                    Y = gbest_X - E * np.abs(J * gbest_X - X[i])
                    Z = Y + np.random.rand(dim) * _levy_hho(dim)
                    X[i] = (Y if func(Y) < func(Z) else Z)

                else:
                    # Hard besiege with progressive rapid dives
                    Y = gbest_X - E * np.abs(J * gbest_X - np.mean(X, axis=0))
                    Z = Y + np.random.rand(dim) * _levy_hho(dim)
                    X[i] = (Y if func(Y) < func(Z) else Z)

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def _levy_hho(dim, beta=1.5):
    """Lévy flight helper for HHO."""
    from scipy.special import gamma
    sigma = (gamma(1+beta) * np.sin(np.pi*beta/2) /
             (gamma((1+beta)/2) * beta * 2**((beta-1)/2)))**(1/beta)
    u = np.random.normal(0, sigma, dim)
    v = np.random.normal(0, 1, dim)
    return u / (np.abs(v)**(1/beta))


# --- Test ---

def mpa(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Marine Predators Algorithm (Faramarzi et al., 2020)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)

    # Elite matrix — top predator
    Elite   = np.tile(gbest_X, (pop_size, 1))
    convergence = [gbest_f]
    P       = 0.5
    FADs    = 0.2

    for t in range(max_iter):
        CF = (1 - t/max_iter) ** (2*t/max_iter)

        RL = 0.05 * _levy_mpa(pop_size, dim)
        RB = np.random.randn(pop_size, dim)

        for i in range(pop_size):
            r  = np.random.rand()
            R  = np.random.rand(dim)

            if t < max_iter / 3:
                # Phase 1 — High velocity ratio (prey moves faster)
                stepsize   = RB[i] * (Elite[i] - RB[i] * X[i])
                X[i]      += P * stepsize

            elif t < 2 * max_iter / 3:
                if i < pop_size // 2:
                    # Phase 2a — Unit velocity ratio (Lévy)
                    stepsize = RL[i] * (Elite[i] - RL[i] * X[i])
                    X[i]    += P * stepsize
                else:
                    # Phase 2b — Unit velocity ratio (Brownian)
                    stepsize = RB[i] * (RB[i] * Elite[i] - X[i])
                    X[i]    += P * CF * stepsize
            else:
                # Phase 3 — Low velocity ratio (predator moves faster)
                stepsize   = RL[i] * (RL[i] * Elite[i] - X[i])
                X[i]      += P * CF * stepsize

            # FADs effect
            if np.random.rand() < FADs:
                U    = np.random.rand(dim) < FADs
                X[i]+= CF * (lb + np.random.rand(dim)*(ub-lb)) * U

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        # Update elite matrix
        Elite = np.tile(gbest_X, (pop_size, 1))
        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def _levy_mpa(n, d, beta=1.5):
    """Lévy flight helper for MPA."""
    from scipy.special import gamma
    sigma = (gamma(1+beta) * np.sin(np.pi*beta/2) /
             (gamma((1+beta)/2) * beta * 2**((beta-1)/2)))**(1/beta)
    u = np.random.normal(0, sigma, (n, d))
    v = np.random.normal(0, 1, (n, d))
    return u / (np.abs(v)**(1/beta))


# --- Test ---

def bfo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        n_swim=4, n_tumble=4,
        seed=42):
    """
    Bacterial Foraging Optimisation (Passino, 2002)

    Parameters:
    -----------
    n_swim   : int — swim steps per chemotaxis (default 4)
    n_tumble : int — tumble steps (default 4)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    step_size = 0.1 * (ub - lb)
    iters_per_cycle = max(1, max_iter // (n_tumble * n_swim + 1))

    for t in range(max_iter):
        for i in range(pop_size):
            # ── Tumble — random direction ─────────────────────
            delta = np.random.randn(dim)
            delta /= (np.linalg.norm(delta) + 1e-10)

            # ── Swim — move in tumble direction ───────────────
            for s in range(n_swim):
                X_new    = X[i] + step_size * delta
                X_new    = np.clip(X_new, lb, ub)
                f_new    = func(X_new)

                if f_new < fitness[i]:
                    X[i]       = X_new
                    fitness[i] = f_new
                    if f_new < gbest_f:
                        gbest_f = f_new
                        gbest_X = X_new.copy()
                else:
                    break

        # ── Reproduction — top half survives ─────────────────
        if t % iters_per_cycle == 0:
            sorted_idx   = np.argsort(fitness)
            X            = np.vstack([
                               X[sorted_idx[:pop_size//2]],
                               X[sorted_idx[:pop_size//2]]
                           ])
            fitness      = np.concatenate([
                               fitness[sorted_idx[:pop_size//2]],
                               fitness[sorted_idx[:pop_size//2]]
                           ])

        # Decrease step size over time
        step_size *= 0.99

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def qbso(func, lb, ub, dim,
         pop_size=30, max_iter=500,
         qs_threshold=0.5,
         seed=42):
    """
    Quorum Sensing Bacterial Swarm Optimisation (QBSO)
    Based on: Li et al. (2019)

    QS used as enhancement to bacterial swarm —
    NOT as standalone framework (key distinction from QSO)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    step_size = 0.1 * (ub - lb)

    for t in range(max_iter):
        # ── Compute quorum signal ─────────────────────────────
        f_worst = np.max(fitness)
        f_best  = np.min(fitness)
        epsilon = 1e-10

        if f_worst - f_best < epsilon:
            qs_signal = 0.5
        else:
            qs_signal = np.mean(
                (f_worst - fitness) / (f_worst - f_best + epsilon))

        for i in range(pop_size):
            delta = np.random.randn(dim)
            delta /= (np.linalg.norm(delta) + 1e-10)

            if qs_signal >= qs_threshold:
                # QS triggered — move toward global best
                direction = gbest_X - X[i]
                norm      = np.linalg.norm(direction) + 1e-10
                X[i]     += step_size * (direction/norm)
            else:
                # QS not triggered — random walk
                X[i]     += step_size * delta

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        step_size *= 0.995
        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def qbho(func, lb, ub, dim,
         pop_size=30, max_iter=500,
         qs_threshold=0.5,
         seed=42):
    """
    Quorum Sensing Bacterial Horde Optimisation (QBHO)
    Based on: Alzaqebah et al. (2023)

    QS used to identify optimal bacterial positions —
    NOT as standalone framework (key distinction from QSO)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        # ── Quorum detection ──────────────────────────────────
        f_worst   = np.max(fitness)
        f_best    = np.min(fitness)
        epsilon   = 1e-10

        qs_signal = np.mean(
            (f_worst - fitness) / (f_worst - f_best + epsilon + 1e-10))

        # ── Worst position used as reference (per QBHO paper) ─
        worst_idx = np.argmax(fitness)

        for i in range(pop_size):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            if qs_signal >= qs_threshold:
                # Quorum active — avoid worst, move to best
                X[i] = (X[i]
                        + r1 * (gbest_X - X[i])
                        - r2 * (X[worst_idx] - X[i]))
            else:
                # Quorum inactive — standard foraging
                rand_X = X[np.random.randint(pop_size)]
                X[i]   = X[i] + r1 * (rand_X - X[i])

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Tests ---

def dbo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Dung Beetle Optimisation (Xue & Shen, 2022)

    Four beetle roles:
    - Ball-rollers  : navigate using celestial cues (exploration)
    - Dancers       : reorient when lost (escape local optima)
    - Foragers      : search near best site (exploitation)
    - Brood-stealers: compete for best positions (intensification)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    # Population split into 4 roles
    n_rollers  = pop_size // 4
    n_dancers  = pop_size // 4
    n_foragers = pop_size // 4
    n_thieves  = pop_size - n_rollers - n_dancers - n_foragers

    # Role index boundaries
    r_end = n_rollers
    d_end = n_rollers + n_dancers
    f_end = n_rollers + n_dancers + n_foragers

    for t in range(max_iter):
        R  = 1 - t / max_iter       # Decreasing radius
        CF = (1 - t/max_iter) ** 2  # Convergence factor

        # ── Ball-rolling beetles (exploration) ────────────────────
        for i in range(r_end):
            if np.random.rand() > 0.9:
                # Dancing reorientation
                X[i] = X[i] + np.tan(
                    np.random.rand(dim)) * np.abs(X[i] - gbest_X)
            else:
                # Navigate toward best with decreasing radius
                r1   = np.random.rand(dim)
                X[i] = X[i] + R * r1 * (gbest_X - X[i])
            X[i] = np.clip(X[i], lb, ub)

        # ── Dancing beetles (escape local optima) ─────────────────
        for i in range(r_end, d_end):
            r1   = np.random.rand(dim)
            X[i] = gbest_X + r1 * np.abs(X[i] - gbest_X) * CF
            X[i] = np.clip(X[i], lb, ub)

        # ── Foraging beetles (exploitation) — FIXED ───────────────
        for i in range(d_end, f_end):
            r1   = np.random.rand(dim)
            r2   = np.random.rand(dim)
            # Move toward global best with random perturbation
            X[i] = (X[i]
                    + r1 * (gbest_X - X[i])
                    + r2 * CF * np.random.randn(dim))
            X[i] = np.clip(X[i], lb, ub)

        # ── Brood-stealing beetles (intensification) ──────────────
        for i in range(f_end, pop_size):
            r1   = np.random.rand(dim)
            r2   = np.random.rand(dim)
            # Steal position near global best
            X[i] = (gbest_X
                    + r1 * CF * (X[i] - gbest_X)
                    + r2 * np.random.randn(dim) * R)
            X[i] = np.clip(X[i], lb, ub)

        # ── Evaluate & update best ────────────────────────────────
        fitness = evaluate_population(func, X)

        # Elite preservation
        worst_idx = np.argmax(fitness)
        if fitness[worst_idx] > gbest_f:
            X[worst_idx]       = gbest_X.copy()
            fitness[worst_idx] = gbest_f

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def poa(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Pelican Optimisation Algorithm (Trojovský & Dehghani, 2022)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        for i in range(pop_size):
            # ── Phase 1: Moving toward prey ───────────────────
            # Random prey selection
            prey_idx  = np.random.randint(pop_size)
            prey_X    = X[prey_idx]
            prey_f    = fitness[prey_idx]

            X1 = X[i] + np.random.rand(dim) * (
                prey_X - np.random.randint(1, 3) * X[i])
            X1 = np.clip(X1, lb, ub)
            f1 = func(X1)

            if f1 < fitness[i]:
                X[i]       = X1
                fitness[i] = f1

            # ── Phase 2: Winging on water surface ─────────────
            R    = 0.2 * (1 - t / max_iter)
            X2   = X[i] + R * (2 * np.random.rand(dim) - 1) * X[i]
            X2   = np.clip(X2, lb, ub)
            f2   = func(X2)

            if f2 < fitness[i]:
                X[i]       = X2
                fitness[i] = f2

            if fitness[i] < gbest_f:
                gbest_f = fitness[i]
                gbest_X = X[i].copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def evo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Electric Eel Foraging Optimiser (EVO)
    Based on: Wang et al. (2024)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        a = 2 * (1 - t / max_iter)  # Decreasing factor

        for i in range(pop_size):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            # ── Electric discharge hunting ────────────────────
            if np.random.rand() < 0.5:
                # Discharge toward best
                X[i] = (X[i]
                        + a * r1 * (gbest_X - X[i])
                        + (1-a) * r2 * (
                            X[np.random.randint(pop_size)] - X[i]))
            else:
                # Passive drift with random component
                beta   = np.random.randn(dim)
                X[i]   = (gbest_X
                          + beta * np.abs(gbest_X - X[i]) * (1 - a))

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Tests ---

def _levy_gjo(n, d, beta=1.5):
    """Lévy flight helper for GJO."""
    from scipy.special import gamma
    sigma = (gamma(1+beta) * np.sin(np.pi*beta/2) /
             (gamma((1+beta)/2) * beta
              * 2**((beta-1)/2)))**(1/beta)
    u = np.random.normal(0, sigma, (n, d))
    v = np.random.normal(0, 1, (n, d))
    return u / (np.abs(v)**(1/beta))


def gjo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Golden Jackal Optimizer (Chopra & Ansari, 2022)
    Published: Expert Systems with Applications, 198, 116924

    Models male and female jackal hunting behaviour:
    - Male jackal: tracks prey (global best)
    - Female jackal: supports male (second best)
    - Prey escape energy decreases over iterations
    """
    X, lb, ub = initialise_population(
                    pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    # Male and female jackal (best two solutions)
    sorted_idx = np.argsort(fitness)
    male_pos   = X[sorted_idx[0]].copy()
    male_f     = fitness[sorted_idx[0]]
    female_pos = X[sorted_idx[1]].copy()
    female_f   = fitness[sorted_idx[1]]

    gbest_f     = male_f
    gbest_X     = male_pos.copy()
    convergence = [gbest_f]

    for t in range(max_iter):
        E1 = 1.5 * (1 - t / max_iter)
        RL = 0.05 * _levy_gjo(pop_size, dim)

        for i in range(pop_size):
            E0 = 2 * np.random.rand() - 1
            E  = E1 * E0

            # Update toward male jackal
            D_male   = np.abs(RL[i] * male_pos - X[i])
            X1       = male_pos - E * D_male

            # Update toward female jackal
            D_female = np.abs(RL[i] * female_pos - X[i])
            X2       = female_pos - E * D_female

            # Average of both updates
            X[i] = np.clip((X1 + X2) / 2, lb, ub)

        fitness = evaluate_population(func, X)

        # Update male and female jackals
        sorted_idx = np.argsort(fitness)

        if fitness[sorted_idx[0]] < male_f:
            male_f   = fitness[sorted_idx[0]]
            male_pos = X[sorted_idx[0]].copy()

        if fitness[sorted_idx[1]] < female_f:
            female_f   = fitness[sorted_idx[1]]
            female_pos = X[sorted_idx[1]].copy()

        if male_f < gbest_f:
            gbest_f = male_f
            gbest_X = male_pos.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---


In [ ]:
# === 6. Official CEC 2017 via opfunu ===
from opfunu.cec_based import cec2017

_FC = {}
def make_func(fid, D):
    if (fid, D) not in _FC:
        _FC[(fid, D)] = getattr(cec2017, f'F{fid}2017')(ndim=D)
    return _FC[(fid, D)]

f = make_func(1, 30)
print('F1 at origin:', f.evaluate(np.zeros(30)))
print('bounds:', f.lb[0], f.ub[0])
print()
print('NOTE: these are the official shift vectors and rotation matrices.')
print('Your make_cec2017_fast() generates its own with np.random.seed(2017),')
print('so numbers will not match your existing results — that is the point.')

In [ ]:
# === 7. Runner + checkpointing ===

RUNNERS = {
    'QSO-time':       lambda *a, **k: qso(*a, theta_mode='time', **k),
    'QSO-stagnation': lambda *a, **k: qso(*a, theta_mode='stagnation', **k),
    'QSO-diversity':  lambda *a, **k: qso(*a, theta_mode='diversity', **k),
    'QSO-clip':       lambda *a, **k: qso(*a, theta_mode='time', bound_mode='clip', **k),
    'PSO': pso, 'GA': ga, 'DE': de, 'GWO': gwo, 'WOA': woa, 'SCA': sca,
    'HHO': hho, 'MPA': mpa, 'BFO': bfo, 'QBSO': qbso, 'QBHO': qbho,
    'DBO': dbo, 'POA': poa, 'EVO': evo, 'GJO': gjo,
}

FIELDS = ['algorithm','fid','dim','max_iter','fes','seed','fbest','secs']

def run_task(task):
    alg, fid, dim, mi, seed = task
    fn = make_func(fid, dim)
    obj = fn.evaluate
    lb, ub = float(fn.lb[0]), float(fn.ub[0])
    t0 = time.time()
    out = RUNNERS[alg](obj, lb, ub, dim, pop_size=POP_SIZE, max_iter=mi, seed=seed)
    secs = time.time() - t0
    fbest = float(out[0])
    traj = []
    if alg.startswith('QSO') and seed == TRAJ_SEED and len(out) >= 7:
        # (convergence, diversity, quorum, phase, theta) subsampled
        step = max(1, len(out[4]) // 400)
        traj = list(zip(range(0, len(out[4]), step),
                        [float(v) for v in out[4][::step]],
                        [float(v) for v in out[6][::step]],
                        [int(v)   for v in out[5][::step]]))
    return (dict(algorithm=alg, fid=fid, dim=dim, max_iter=mi,
                 fes=POP_SIZE*(mi+1), seed=seed, fbest=fbest, secs=secs), traj)

def load_done():
    if not os.path.exists(RESULTS): return set()
    d = pd.read_csv(RESULTS)
    return set(zip(d.algorithm, d.fid, d.dim, d.max_iter, d.seed))

def append(rows):
    new = not os.path.exists(RESULTS)
    with open(RESULTS, 'a', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=FIELDS)
        if new: w.writeheader()
        for r in rows: w.writerow(r)
        fh.flush(); os.fsync(fh.fileno())

ALL = [(a, f, d, mi, s)
       for d in DIMS for mi in iters_for(d)
       for f in FUNCTIONS for a in ALGORITHMS for s in SEEDS]
ALL.sort(key=lambda t: (t[2], t[1]))     # 30D first, then by function
print(len(ALL), 'total tasks')
for d in DIMS:
    print(f'  D={d}: max_iter budgets {iters_for(d)} '
          f'-> FEs {[POP_SIZE*(m+1) for m in iters_for(d)]}')

In [ ]:
# === 8. TIMING PROBE — run before cell 9 ===
probe = []
for a in ALGORITHMS:
    t0 = time.time(); run_task((a, 1, 30, 500, 42)); probe.append((a, time.time()-t0))
for a, s in probe: print(f'{a:16s} {s:6.2f}s @ max_iter=500, D=30')

per_iter = np.mean([s for _, s in probe]) / 500
tot_iters = sum(mi for d in DIMS for mi in iters_for(d)) * len(FUNCTIONS) * len(ALGORITHMS) * len(SEEDS)
h = tot_iters * per_iter / 3600
print(f'\nprojected ~{h:.1f} core-hours -> ~{h/max(1,N_JOBS):.1f} h on {N_JOBS} cores')
print('If too large: cut SEEDS, FUNCTIONS, or ALGORITHMS in cell 3.')

In [ ]:
# === 9. Main loop (resumable) ===
done = load_done()
todo = [t for t in ALL if t not in done]
print(f'{len(todo)} remaining of {len(ALL)}')

t0 = time.time()
for i in range(0, len(todo), BATCH):
    batch = todo[i:i+BATCH]; rows = []
    with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
        for row, traj in ex.map(run_task, batch):
            rows.append(row)
            if traj:
                nm = f"traj_{row['algorithm']}_F{row['fid']}_D{row['dim']}_I{row['max_iter']}.json"
                json.dump(traj, open(os.path.join(TRAJ_DIR, nm), 'w'))
    append(rows)
    el = time.time()-t0; fr = (i+len(batch))/len(todo)
    print(f"{i+len(batch):>6}/{len(todo)} ({100*fr:5.1f}%) "
          f"elapsed {el/60:6.1f}m eta {el/max(fr,1e-9)*(1-fr)/60:6.1f}m", flush=True)
print('DONE')

In [ ]:
# === 10. Analysis ===
from scipy.stats import rankdata, friedmanchisquare

df = pd.read_csv(RESULTS)
rows = []
for dim in sorted(df.dim.unique()):
    for mi in sorted(df[df.dim==dim].max_iter.unique()):
        sub = df[(df.dim==dim)&(df.max_iter==mi)]
        piv = sub.groupby(['fid','algorithm']).fbest.mean().unstack().dropna(axis=1)
        if piv.shape[1] < 2: continue
        R = np.apply_along_axis(rankdata, 1, piv.values)
        mfr = pd.Series(R.mean(axis=0), index=piv.columns).sort_values()
        for a, r in mfr.items():
            rows.append(dict(dim=dim, max_iter=mi, fes=POP_SIZE*(mi+1),
                             algorithm=a, mfr=round(float(r),3),
                             pos=int(mfr.index.get_loc(a))+1))
S = pd.DataFrame(rows)
S.to_csv(os.path.join(OUT,'summary_ranks.csv'), index=False)

print('MEAN FRIEDMAN RANK (lower = better)\n')
print(S.pivot_table(index='algorithm', columns=['dim','fes'], values='mfr').round(2).to_string())

In [ ]:
# === 11. The three questions ===
print('Q1 — DOES QSO RANK HOLD AS BUDGET GROWS?\n')
print(S[S.algorithm=='QSO-time'][['dim','fes','pos','mfr']]
        .sort_values(['dim','fes']).to_string(index=False))
print('\n  stable position -> scalability claim survives')
print('  position worsens as FEs grow -> 50D result is a budget artifact\n')

print('Q2 — IS A STATE-DEPENDENT THRESHOLD BETTER?\n')
print(S[S.algorithm.str.startswith('QSO')]
        .pivot_table(index='algorithm', columns=['dim','fes'], values='mfr')
        .round(2).to_string())
print('\n  if either variant wins, replace Eq.4 — it also makes the')
print('  Section 1 novelty claim true as written\n')

print('Q4 — IS THE EFFECT THE QUORUM SIGNAL, OR THE BOUNDARY REPAIR?\n')
print(S[S.algorithm.isin(['QSO-time','QSO-clip'])]
        .pivot_table(index='algorithm', columns=['dim','fes'], values='mfr')
        .round(2).to_string())
print('\n  QSO-clip differs from QSO-time ONLY in boundary handling.')
print('  If QSO-time beats QSO-clip and the gap WIDENS from 30D to 50D, the')
print('  dimensional advantage comes from clip_to_bounds() injecting diversity,')
print('  not from the aggregated quorum signal. That is the rival explanation.\n')

print('Q3 — OFFICIAL vs SELF-GENERATED CEC 2017')
print('  compare the max_iter=500 column against Table 3 in the manuscript.')
print('  large divergence means the published numbers rest on functions that')
print('  are not the CEC 2017 suite.')

## After the run

**Trajectories.** `diagnostics/trajectories/*.json` holds `(iteration, C, theta, phase)`.
Your `qso()` already returns `quorum_history`, `theta_history`, `phase_history` and
`diversity` — so the switching figure the manuscript lacks needs no new experiments at
all, just a plot of what you are already logging.

**If Q1 comes back badly** — QSO's rank degrading as the budget grows — that is still a
publishable result, and a more interesting one than another algorithm ranking: it would
show that CEC 2017 comparisons at truncated budgets measure convergence speed rather than
solution quality, which affects a large body of published work.

**Scaling up.** Set `SEEDS = list(range(42,72))` and switch `ALGORITHMS` to the full
17-entry list in cell 3, then rerun cell 9. Completed runs are skipped.